In [0]:
# Camada gold: df_logs é o resultado das transformações da camada silver
from pyspark.sql.functions import avg, col, count, sum

df_logs = (
    spark.table("streaming_dev.silver.logs_streaming")
    .drop("ingest_timestamp", "source_file")
)

# Agregações dataframe

df_logs.groupBy("subscription_type") \
    .agg(
        count("*").alias("total_sessoes"),
        avg("watch_time_minutes").alias("media_minutos_assistidos"),
        sum("watch_time_minutes").alias("total_minutos_assistidos")
    ) \
    .display()

df_logs.groupBy("subscription_type") \
    .agg(
        count("*").alias("total_sessoes"),
        avg("watch_time_minutes").alias("media_minutos_assistidos")
    ) \
    .orderBy(col("total_sessoes").desc()) \
    .display()

df_logs.groupBy("country") \
    .agg(
        count("*").alias("total_sessoes"),
        avg("watch_time_minutes").alias("media_minutos_assistidos")
    ) \
    .orderBy(col("total_sessoes").desc()) \
    .display()

In [0]:
df_catalogo = (
    spark.table("streaming_dev.silver.catalogo_filmes")
    .drop("ingest_timestamp", "source_file")
)

df_usuarios = (
    spark.table("streaming_dev.silver.usuarios")
    .drop("ingest_timestamp", "source_file")
    .withColumnRenamed("country", "user_country")
    .withColumnRenamed("subscription_type", "user_subscription_type")
)

df_enriquecido = df_logs.join(
    df_catalogo,
    on="movie_id",
    how="left"
)

df_final = df_enriquecido.join(
    df_usuarios,
    on="user_id",
    how="left"
)


display(df_final.limit(5))

In [0]:
# Gravando o DataFrame enriquecido na camada gold


df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("streaming_dev.gold.logs_enriquecidos")